# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abd481/Search-Ranks-/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

*One row represents the daily performance of one content page for one client, using data from March 2026.*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features
- `impressions`: observed search visibility.
- `clicks`: observed search clicks.
- `sessions`: observed website sessions.
- `avg_position`: observed search ranking position.
- `content_age_days`: how old the content is.
- `days_since_last_update`: how long since the content was last updated.

### Label / Proxy
- `is_declining_label`: a proxy label indicating whether the page is currently classified as declining. It is a starter proxy, not a true future outcome.

### Context
- `client_hash_id`: identifies the client for grouping and validation.
- `content_hash_id`: identifies the content page.
- `report_date`: identifies the date of the observation.

### Excluded
- Product decision scores or flags such as `priority_score`, `health_score`, and `action_type` are excluded because they represent existing product decisions and could cause the model to simply reproduce those decisions.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Grain verification

For March 2026, the slice contains 9,841,378 rows across 55 clients, 331,437 content pages, and 31 dates.

The number of unique client-content-date combinations is also 9,841,378, which matches the total row count. This supports my definition that one row represents the daily performance of one content page for one client.

### Row count and date range verification

The March 2026 slice contains 9,841,378 rows, with data ranging from March 1 to March 31, 2026. This confirms that the selected slice covers the full month used for my analysis.

### Data availability verification

In the March 2026 slice, there are 9,841,378 total rows. GSC data is available for 3,611,061 rows, while GA4 data is available for 413,966 rows. Both GSC and GA4 data are available together for 364,347 rows.

This shows that data availability varies across the warehouse, so my feature selection and scoring approach should account for missing signals rather than assuming every page has complete data.

In [17]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

In [18]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows
fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS content_pages,
    COUNT(DISTINCT report_date) AS dates,
    COUNT(DISTINCT
        CONCAT(
            client_hash_id, '|',
            content_hash_id, '|',
            CAST(report_date AS VARCHAR)
        )
    ) AS unique_client_content_date
FROM {TABLES['fact_daily']}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
"""

con.sql(query).show()


┌────────────────────┬─────────────┐
│    column_name     │ column_type │
│      varchar       │   varchar   │
├────────────────────┼─────────────┤
│ report_date        │ DATE        │
│ client_hash_id     │ VARCHAR     │
│ content_hash_id    │ VARCHAR     │
│ client_has_gsc     │ BOOLEAN     │
│ client_has_ga4     │ BOOLEAN     │
│ gsc_data_available │ BOOLEAN     │
│ ga4_data_available │ BOOLEAN     │
│ gsc_impressions    │ BIGINT      │
│ gsc_clicks         │ BIGINT      │
│ gsc_sum_position   │ BIGINT      │
│      ·             │   ·         │
│      ·             │   ·         │
│      ·             │   ·         │
│ sessions_ai        │ BIGINT      │
│ ai_chatgpt         │ BIGINT      │
│ ai_perplexity      │ BIGINT      │
│ ai_gemini          │ BIGINT      │
│ ai_copilot         │ BIGINT      │
│ ai_claude          │ BIGINT      │
│ ai_meta            │ BIGINT      │
│ ai_other           │ BIGINT      │
│ scroll_events      │ BIGINT      │
│ month              │ VARCHAR     │
├

In [ ]:
query = f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM {TABLES['fact_daily']}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
"""

con.sql(query).show()

┌───────────┬────────────┬────────────┐
│ row_count │  min_date  │  max_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘



In [ ]:
query = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_rows,
    COUNT(*) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS ga4_available_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
        AND ga4_data_available IS TRUE
    ) AS both_available_rows
FROM {TABLES['fact_daily']}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
"""

con.sql(query).show()

┌────────────┬────────────────────┬────────────────────┬─────────────────────┐
│ total_rows │ gsc_available_rows │ ga4_available_rows │ both_available_rows │
│   int64    │       int64        │       int64        │        int64        │
├────────────┼────────────────────┼────────────────────┼─────────────────────┤
│    9841378 │            3611061 │             413966 │              364347 │
└────────────┴────────────────────┴────────────────────┴─────────────────────┘



In [ ]:
con.sql(f"""
SELECT column_name, column_type
FROM (DESCRIBE SELECT * FROM {TABLES['dim_content']})
""").show()

┌────────────────────────────┬─────────────┐
│        column_name         │ column_type │
│          varchar           │   varchar   │
├────────────────────────────┼─────────────┤
│ client_hash_id             │ VARCHAR     │
│ content_hash_id            │ VARCHAR     │
│ keyword_hash_id            │ VARCHAR     │
│ url_hash_id                │ VARCHAR     │
│ keyword_char_count         │ BIGINT      │
│ keyword_token_count        │ BIGINT      │
│ url_char_count             │ BIGINT      │
│ content_created_date       │ DATE        │
│ content_updated_date       │ DATE        │
│ content_type               │ VARCHAR     │
│      ·                     │   ·         │
│      ·                     │   ·         │
│      ·                     │   ·         │
│ category_count             │ BIGINT      │
│ keyword_created_date       │ DATE        │
│ provider_used              │ VARCHAR     │
│ model_used                 │ VARCHAR     │
│ char_count                 │ BIGINT      │
│ word_cou

In [ ]:
feature_query = f"""
SELECT
    content_hash_id,

    SUM(gsc_impressions) AS impressions_30d,

    SUM(gsc_clicks) AS clicks_30d,

    CASE
        WHEN SUM(gsc_impressions) > 0
        THEN SUM(gsc_sum_position) * 1.0 / SUM(gsc_impressions)
        ELSE NULL
    END AS avg_position,

    SUM(
        COALESCE(sessions_ai, 0)
        + COALESCE(sessions_paid, 0)
        + COALESCE(sessions_social, 0)
        + COALESCE(sessions_direct, 0)
        + COALESCE(sessions_organic, 0)
    ) AS sessions_30d

FROM {TABLES['fact_daily']}

WHERE report_date >= DATE '2026-03-01'
  AND report_date <= DATE '2026-03-31'

GROUP BY content_hash_id
"""

features = con.sql(feature_query).df()

features.head()

,content_hash_id,impressions_30d,clicks_30d,avg_position,sessions_30d
0,content_1121cd4a6ab03fd6,0.0,0.0,NaN,0.0
1,content_3da1478be246c0f2,0.0,0.0,NaN,0.0
2,content_4e8ea3bc30a032dc,0.0,0.0,NaN,0.0
3,content_f08a29ae14406c38,0.0,0.0,NaN,0.0
4,content_a3ce58ada9ec86c6,0.0,0.0,NaN,0.0


In [ ]:
age_query = f"""
SELECT
    content_hash_id,
    DATE_DIFF(
        'day',
        content_created_date,
        DATE '2026-03-31'
    ) AS content_age_days
FROM {TABLES['dim_content']}
WHERE content_created_date IS NOT NULL
"""

content_age = con.sql(age_query).df()

content_age.head()

,content_hash_id,content_age_days
0,content_004de9653278b5a4,-60
1,content_00dc5efae381b2ab,-73
2,content_01410f2556c327ac,-39
3,content_019f27f634053ca7,-76
4,content_01efa71faea45dcc,-51


In [ ]:
features = features.merge(
    content_age,
    on="content_hash_id",
    how="left"
)

features.head()

,content_hash_id,impressions_30d,clicks_30d,avg_position,sessions_30d,content_age_days
0,content_1121cd4a6ab03fd6,0.0,0.0,NaN,0.0,344
1,content_3da1478be246c0f2,0.0,0.0,NaN,0.0,344
2,content_4e8ea3bc30a032dc,0.0,0.0,NaN,0.0,344
3,content_f08a29ae14406c38,0.0,0.0,NaN,0.0,344
4,content_a3ce58ada9ec86c6,0.0,0.0,NaN,0.0,344


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## 4. Data limits

This data can help rank pages for review, but it cannot tell me whether a recommended refresh will actually cause better performance.

The warehouse has an unbalanced history, so different clients and pages may have different amounts of historical data.

Some rows have GSC data available while GA4 data is missing, so engagement-based signals cannot be treated as complete for every row.

The March feature window also cannot be used to make claims about future performance unless a separate future window is defined for the label. Therefore, these features are only used for decision-time ranking and should not include information from after March 31, 2026.

The data is observational, so it can show signals associated with content performance but cannot prove that changing a page will cause traffic or ranking improvements.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.